In [1]:
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date

import pandas as pd
import requests
from dotenv import load_dotenv, find_dotenv
from requests.adapters import HTTPAdapter
from sqlalchemy import create_engine, text
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

load_dotenv(find_dotenv(usecwd=True))
engine = create_engine(os.environ["DATABASE_URL"])

API_URL = "https://api.mfapi.in/mf/{code}"
HISTORY_START = date(2019, 1, 1)     # keep ~7 years of history
MAX_WORKERS = 5                      # be polite to a free community API
BATCH_SIZE = 100                     # save + checkpoint every 100 schemes

In [2]:
DDL = """
CREATE TABLE IF NOT EXISTS raw.nav_history (
    scheme_code  INTEGER       NOT NULL,
    nav_date     DATE          NOT NULL,
    nav          NUMERIC(18,4) NOT NULL,
    loaded_at    TIMESTAMPTZ   NOT NULL DEFAULT now(),
    PRIMARY KEY (scheme_code, nav_date)
);

CREATE TABLE IF NOT EXISTS raw.scheme_meta (
    scheme_code            INTEGER PRIMARY KEY,
    fund_house             TEXT,
    scheme_type            TEXT,
    scheme_category        TEXT,
    scheme_name            TEXT,
    isin_growth            TEXT,
    isin_div_reinvestment  TEXT,
    fetched_at             TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE TABLE IF NOT EXISTS raw.backfill_status (
    scheme_code   INTEGER PRIMARY KEY,
    status        TEXT NOT NULL,          -- success / failed / empty
    rows_loaded   INTEGER,
    first_date    DATE,
    last_date     DATE,
    error         TEXT,
    attempted_at  TIMESTAMPTZ NOT NULL DEFAULT now()
);
"""
with engine.begin() as conn:
    conn.execute(text(DDL))
print("tables ready")

tables ready


In [3]:
schemes = pd.read_sql(text("""
    SELECT scheme_code, scheme_name, plan, option, scheme_category, fund_house
    FROM raw.amfi_nav_daily
    WHERE nav_date >= (SELECT MAX(nav_date) FROM raw.amfi_nav_daily) - 7
      AND scheme_type ILIKE 'Open Ended%'
      AND scheme_category ILIKE 'Equity Scheme%'
      AND option ILIKE '%growth%'
      AND plan IN ('Direct Plan', 'Regular Plan')
"""), engine).drop_duplicates("scheme_code")

print(f"Schemes selected: {len(schemes):,}")
print(schemes["plan"].value_counts())
schemes["scheme_category"].value_counts()

Schemes selected: 1,147
plan
Regular Plan    575
Direct Plan     572
Name: count, dtype: int64


scheme_category
Equity Scheme - Sectoral/ Thematic       341
Equity Schemes - Thematic Fund           120
Equity Scheme - Flexi Cap Fund            60
Equity Scheme - Small Cap Fund            49
Equity Scheme - Large Cap Fund            46
Equity Scheme - Large & Mid Cap Fund      44
Equity Schemes - ELSS- Tax Saver Fund     42
Equity Schemes - Sectoral Fund            40
Equity Scheme - Mid Cap Fund              38
Equity Scheme - Focused Fund              36
Equity Scheme - ELSS                      34
Equity Scheme - Multi Cap Fund            34
Equity Schemes - Flexi Cap Fund           32
Equity Schemes - Multi Cap Fund           31
Equity Scheme - Value Fund                29
Equity Schemes - Large & Mid Cap Fund     28
Equity Schemes - Mid Cap Fund             28
Equity Schemes - Large Cap Fund           22
Equity Schemes - Small Cap Fund           22
Equity Schemes - Focused Fund             20
Equity Scheme - Dividend Yield Fund       18
Equity Schemes - Value Fund            

In [4]:
def http_session():
    session = requests.Session()
    retry = Retry(total=4, backoff_factor=1.5, status_forcelist=[429, 500, 502, 503, 504])
    session.mount("https://", HTTPAdapter(max_retries=retry, pool_maxsize=MAX_WORKERS))
    session.headers["User-Agent"] = "FundLens/1.0 (portfolio data project)"
    return session

session = http_session()
sample = session.get(API_URL.format(code=int(schemes["scheme_code"].iloc[0])), timeout=30).json()
print(sample["meta"])
print(sample["data"][:3])
print("History points:", len(sample["data"]))

{'fund_house': 'LIC Mutual Fund', 'scheme_type': 'Open Ended Schemes', 'scheme_category': 'Equity Scheme - Dividend Yield Fund', 'scheme_code': 152022, 'scheme_name': 'LIC MF Dividend Yield Fund - Direct Plan - Growth', 'isin_growth': 'INF397L01LC1', 'isin_div_reinvestment': None}
[{'date': '12-10-2026', 'nav': '34.61070'}, {'date': '09-10-2026', 'nav': '34.55500'}, {'date': '08-10-2026', 'nav': '34.18600'}]
History points: 787


In [5]:
def fetch_history(code):
    """Download one scheme's NAV history. Returns (meta, DataFrame) or raises."""
    resp = session.get(API_URL.format(code=code), timeout=30)
    resp.raise_for_status()
    payload = resp.json()
    df = pd.DataFrame(payload.get("data") or [])
    if df.empty:
        return payload.get("meta", {}), df
    df["nav_date"] = pd.to_datetime(df["date"], format="%d-%m-%Y").dt.date
    df["nav"] = pd.to_numeric(df["nav"], errors="coerce")
    df = df[(df["nav"] > 0) & (df["nav_date"] >= HISTORY_START)]
    df["scheme_code"] = code
    df = df.drop_duplicates(["scheme_code", "nav_date"])
    return payload.get("meta", {}), df[["scheme_code", "nav_date", "nav"]]

meta, hist = fetch_history(int(schemes["scheme_code"].iloc[0]))
print(meta.get("scheme_name"), "|", len(hist), "rows from", hist["nav_date"].min(), "to", hist["nav_date"].max())

LIC MF Dividend Yield Fund - Direct Plan - Growth | 787 rows from 2023-07-31 to 2026-10-12


In [6]:
def save_batch(histories, metas, statuses):
    with engine.begin() as conn:
        if histories:
            pd.concat(histories).to_sql("tmp_hist", conn, if_exists="replace", index=False)
            conn.execute(text("""
                INSERT INTO raw.nav_history (scheme_code, nav_date, nav)
                SELECT scheme_code, nav_date, nav FROM tmp_hist
                ON CONFLICT (scheme_code, nav_date) DO UPDATE SET nav = EXCLUDED.nav, loaded_at = now()"""))
            conn.execute(text("DROP TABLE tmp_hist"))
        if metas:
            pd.DataFrame(metas).to_sql("tmp_meta", conn, if_exists="replace", index=False)
            conn.execute(text("""
                INSERT INTO raw.scheme_meta (scheme_code, fund_house, scheme_type, scheme_category,
                                             scheme_name, isin_growth, isin_div_reinvestment)
                SELECT scheme_code, fund_house, scheme_type, scheme_category,
                       scheme_name, isin_growth, isin_div_reinvestment FROM tmp_meta
                ON CONFLICT (scheme_code) DO UPDATE SET
                    fund_house = EXCLUDED.fund_house, scheme_type = EXCLUDED.scheme_type,
                    scheme_category = EXCLUDED.scheme_category, scheme_name = EXCLUDED.scheme_name,
                    isin_growth = EXCLUDED.isin_growth,
                    isin_div_reinvestment = EXCLUDED.isin_div_reinvestment, fetched_at = now()"""))
            conn.execute(text("DROP TABLE tmp_meta"))
        pd.DataFrame(statuses).to_sql("tmp_status", conn, if_exists="replace", index=False)
        conn.execute(text("""
            INSERT INTO raw.backfill_status (scheme_code, status, rows_loaded, first_date, last_date, error)
            SELECT scheme_code, status, rows_loaded, first_date::date, last_date::date, error FROM tmp_status
            ON CONFLICT (scheme_code) DO UPDATE SET
                status = EXCLUDED.status, rows_loaded = EXCLUDED.rows_loaded,
                first_date = EXCLUDED.first_date, last_date = EXCLUDED.last_date,
                error = EXCLUDED.error, attempted_at = now()"""))
        conn.execute(text("DROP TABLE tmp_status"))


def run_backfill(codes):
    for start in tqdm(range(0, len(codes), BATCH_SIZE), desc="batches"):
        batch = codes[start:start + BATCH_SIZE]
        histories, metas, statuses = [], [], []
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
            futures = {pool.submit(fetch_history, c): c for c in batch}
            for fut in as_completed(futures):
                code = futures[fut]
                try:
                    meta, df = fut.result()
                    if df.empty:
                        statuses.append({"scheme_code": code, "status": "empty", "rows_loaded": 0,
                                         "first_date": None, "last_date": None, "error": None})
                        continue
                    histories.append(df)
                    metas.append({"scheme_code": code, **{k: meta.get(k) for k in
                                  ["fund_house", "scheme_type", "scheme_category", "scheme_name",
                                   "isin_growth", "isin_div_reinvestment"]}})
                    statuses.append({"scheme_code": code, "status": "success", "rows_loaded": len(df),
                                     "first_date": str(df["nav_date"].min()),
                                     "last_date": str(df["nav_date"].max()), "error": None})
                except Exception as e:
                    statuses.append({"scheme_code": code, "status": "failed", "rows_loaded": 0,
                                     "first_date": None, "last_date": None, "error": str(e)[:300]})
        save_batch(histories, metas, statuses)

In [7]:
pilot = [int(c) for c in schemes["scheme_code"].head(3)]
run_backfill(pilot)
pd.read_sql("SELECT * FROM raw.backfill_status ORDER BY attempted_at DESC", engine)

batches:   0%|          | 0/1 [00:00<?, ?it/s]

,scheme_code,status,rows_loaded,first_date,last_date,error,attempted_at
0,152022,success,787,2023-07-31,2026-10-12,None,2026-10-13 04:54:03.504101+00:00
1,119769,success,1916,2019-01-01,2026-10-12,None,2026-10-13 04:54:03.504101+00:00
2,103040,success,1916,2019-01-01,2026-10-12,None,2026-10-13 04:54:03.504101+00:00


In [8]:
done = set(pd.read_sql("SELECT scheme_code FROM raw.backfill_status WHERE status IN ('success','empty')", engine)["scheme_code"])
todo = [int(c) for c in schemes["scheme_code"] if int(c) not in done]
print(f"Already done: {len(done):,} | To fetch now: {len(todo):,}")

t0 = time.time()
run_backfill(todo)
print(f"Finished in {(time.time() - t0) / 60:.1f} min")

Already done: 3 | To fetch now: 1,144


batches:   0%|          | 0/12 [00:00<?, ?it/s]

Finished in 1.3 min


In [ ]:
pd.read_sql("SELECT status, COUNT(*) AS schemes, SUM(rows_loaded) AS rows FROM raw.backfill_status GROUP BY status", engine)

In [ ]:
pd.read_sql("""
    SELECT COUNT(*) AS rows, COUNT(DISTINCT scheme_code) AS schemes,
           MIN(nav_date) AS first_date, MAX(nav_date) AS last_date
    FROM raw.nav_history
""", engine)

In [ ]:
pd.read_sql("""
    WITH moves AS (
        SELECT scheme_code, nav_date, nav,
               LAG(nav) OVER (PARTITION BY scheme_code ORDER BY nav_date) AS prev_nav
        FROM raw.nav_history
    )
    SELECT m.scheme_code, s.scheme_name, m.nav_date, m.prev_nav, m.nav,
           ROUND((m.nav / m.prev_nav - 1) * 100, 2) AS pct_change
    FROM moves m JOIN raw.scheme_meta s USING (scheme_code)
    WHERE m.prev_nav IS NOT NULL AND ABS(m.nav / m.prev_nav - 1) > 0.10
    ORDER BY ABS(m.nav / m.prev_nav - 1) DESC
    LIMIT 15
""", engine)